# CTL / Learning mFISH metadata tables

Builds the two metadata tables the problem sets use to choose a session, in the same
pattern as `V1DD_metadata.ipynb` and `bci_metadata.ipynb`.

| output | one row per |
| --- | --- |
| `ctl_session_metadata.csv` | session |
| `ctl_plane_metadata.csv` | session x imaging plane |

Both are written to `/code/metadata/`. Run this notebook when a new processing batch
lands; the problem sets read the CSVs, not docDB.


In [1]:
import os
import time
from datetime import datetime

import numpy as np
import pandas as pd

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)

OUTPUT_DIR = '/code/metadata'
DATA_DIR = '/data'
CAPSULE_MOUNT = 'cell-types-and-learning-ophys-NWBs'   # attached asset with the NWBs

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [2]:
from aind_data_access_api.document_db import MetadataDbClient

API_GATEWAY_HOST = "api.allenneuraldynamics.org"
DATABASE = 'metadata_index'
COLLECTION = 'data_assets'

docdb_api_client = MetadataDbClient(
   host=API_GATEWAY_HOST,
   database=DATABASE,
   collection=COLLECTION,
)
print(docdb_api_client._base_url)


https://api.allenneuraldynamics.org/v1/metadata_index/data_assets


#### The gateway is flaky -- wrap every aggregation

`api.allenneuraldynamics.org` returns intermittent **503 Service Unavailable**, and
**400 Bad Request** on larger pipelines. A bare `aggregate_docdb_records` call will
fail partway through and lose the batch, so every aggregation below goes through
`agg()` and every pipeline is scoped to one mouse.


In [3]:
def agg(pipeline, tries=4, base_sleep=5):
    """Run a docDB aggregation, retrying the gateway's intermittent 503s."""
    last = None
    for attempt in range(tries):
        try:
            return docdb_api_client.aggregate_docdb_records(pipeline=pipeline)
        except Exception as exc:
            last = exc
            if attempt < tries - 1:
                time.sleep(base_sleep * (attempt + 1))
    raise RuntimeError(f'docDB aggregate failed after {tries} attempts: {last}')


# The cohort is defined by subject, NOT by project_name: five different
# project_name values are interleaved across the same mice.
CTL_MICE = ['782149', '790322', '788406', '800792', '800995', '804363']

# Processed asset names end in _processed_<date>_<time>. Anchor at end-of-string --
# `_processed_` also appears mid-name in further-derived assets (coreg, czstack).
PROCESSED_PATTERN = (r'^multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+'
                     r'_processed_\d{4}-\d{2}-\d{2}_[\d-]+$')


## Session table

**Use the v1 `session.*` paths.** This cohort has never been migrated to
aind-data-schema v2, so `acquisition` is present but null on every document. Querying
`acquisition.acquisition_type` returns `None` for every row and the column quietly
vanishes from the table rather than raising.

`{"session": {"$type": "object"}}` is the correct generation guard. `$ifNull` does not
work here: `acquisition` exists as a null field on every document, so a guard built on
it never fires.


In [4]:
def session_pipeline(subject_id):
    return [
        {'$match': {'data_description.subject_id': subject_id,
                    'name': {'$regex': '_processed_'},
                    'session': {'$type': 'object'}}},
        {'$project': {
            'name': 1,
            'subject_id': '$data_description.subject_id',
            'project_name': '$data_description.project_name',
            'session_type': '$session.session_type',
            'session_start_time': '$session.session_start_time',
            'session_end_time': '$session.session_end_time',
            'rig': '$session.rig_id',
            'genotype': '$subject.genotype',
            'sex': '$subject.sex',
            'date_of_birth': '$subject.date_of_birth',
        }},
    ]


records = []
for mouse in CTL_MICE:
    rows = agg(session_pipeline(mouse))
    records.extend(rows)
    print(f'{mouse}: {len(rows)} processed assets')

sessions = pd.DataFrame(records)
print(f'\n{len(sessions)} rows before deduplication')


RuntimeError: docDB aggregate failed after 4 attempts: 503 Server Error: Service Unavailable for url: https://api.allenneuraldynamics.org/v1/metadata_index/data_assets/aggregate

### One row per session, newest processing only

A session is reprocessed whenever the pipeline changes, so the same session appears
several times under different `_processed_` stamps. Keep the newest.


In [ ]:
sessions = sessions[sessions.name.str.match(PROCESSED_PATTERN)].copy()

sessions['session_id'] = sessions.name.str.extract(
    r'^(multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+)_processed_')
sessions['processed_stamp'] = sessions.name.str.extract(
    r'_processed_(\d{4}-\d{2}-\d{2}_[\d-]+)$')

sessions = (sessions.sort_values('processed_stamp')
                    .drop_duplicates('session_id', keep='last'))

print(f'{len(sessions)} unique sessions across {sessions.subject_id.nunique()} mice')
print(sessions.subject_id.value_counts().sort_index().to_string())


In [ ]:
# Dates and ages, as in the other metadata notebooks
sessions['session_date'] = sessions.session_start_time.map(
    lambda x: datetime.fromisoformat(x).date())
sessions['session_time'] = sessions.session_start_time.map(
    lambda x: datetime.fromisoformat(x).time())
sessions['date_of_birth'] = sessions.date_of_birth.map(
    lambda x: datetime.strptime(x, '%Y-%m-%d').date() if isinstance(x, str) else x)
sessions['age_days'] = [(d - b).days if pd.notnull(b) else np.nan
                        for d, b in zip(sessions.session_date, sessions.date_of_birth)]

# Training stage and image set, parsed from session_type
sessions['stage'] = sessions.session_type.str.extract(
    r'^(TRAINING_\d|OPHYS_\d|STAGE_\d)')
sessions['image_set'] = sessions.session_type.str.extract(r'_images_([AB])')
sessions['session_number'] = (sessions.sort_values('session_date')
                                      .groupby('subject_id').cumcount() + 1)

order = ['subject_id', 'session_id', 'name', 'session_type', 'stage', 'image_set',
         'session_number', 'session_date', 'session_time', 'age_days',
         'genotype', 'sex', 'date_of_birth', 'rig', 'project_name',
         'processed_stamp', '_id']
sessions = (sessions[[c for c in order if c in sessions.columns]]
            .sort_values(['subject_id', 'session_date'])
            .reset_index(drop=True))
sessions.head(10)


## Plane table

QC and imaging geometry are **per plane**, not per session, so the plane table is what
the problem sets use to pick a plane. It comes from `$unwind`-ing
`session.data_streams` and then `ophys_fovs`.

That double `$unwind` is what triggers the gateway's 503s on the larger mice, so this
runs one mouse at a time and reports which ones failed rather than dying.


In [ ]:
def plane_pipeline(subject_id):
    return [
        {'$match': {'data_description.subject_id': subject_id,
                    'name': {'$regex': '_processed_'},
                    'session': {'$type': 'object'}}},
        {'$unwind': '$session.data_streams'},
        {'$unwind': '$session.data_streams.ophys_fovs'},
        {'$project': {
            'name': 1,
            'subject_id': '$data_description.subject_id',
            'session_type': '$session.session_type',
            'fov_index': '$session.data_streams.ophys_fovs.index',
            'targeted_structure': '$session.data_streams.ophys_fovs.targeted_structure',
            'imaging_depth': '$session.data_streams.ophys_fovs.imaging_depth',
            'frame_rate': '$session.data_streams.ophys_fovs.frame_rate',
            'scanimage_roi_index':
                '$session.data_streams.ophys_fovs.scanimage_roi_index',
        }},
    ]


plane_records, failed = [], []
for mouse in CTL_MICE:
    try:
        rows = agg(plane_pipeline(mouse))
        plane_records.extend(rows)
        print(f'{mouse}: {len(rows)} plane rows')
    except Exception as exc:
        failed.append(mouse)
        print(f'{mouse}: FAILED -- {str(exc)[:80]}')

if failed:
    print(f'\nRETRY THESE: {failed}  (gateway 503s, not a data problem)')

planes = pd.DataFrame(plane_records)
print(f'\n{len(planes)} plane rows')


In [ ]:
# Same dedup as the session table, then join the session columns on
planes = planes[planes.name.str.match(PROCESSED_PATTERN)].copy()
planes['session_id'] = planes.name.str.extract(
    r'^(multiplane-ophys_\d+_\d{4}-\d{2}-\d{2}_[\d-]+)_processed_')
planes['processed_stamp'] = planes.name.str.extract(
    r'_processed_(\d{4}-\d{2}-\d{2}_[\d-]+)$')

# keep only the newest processing generation per session, matching `sessions`
keep = set(zip(sessions.session_id, sessions.processed_stamp))
planes = planes[[(s, p) in keep for s, p in
                 zip(planes.session_id, planes.processed_stamp)]]

planes['plane_name'] = (planes.targeted_structure.astype(str) + '_'
                       + planes.fov_index.astype(str))

planes = planes.merge(
    sessions[['session_id', 'subject_id', 'session_date', 'stage', 'image_set',
              'session_number', 'genotype']],
    on=['session_id', 'subject_id'], how='left', suffixes=('', '_session'))

order = ['subject_id', 'session_id', 'session_type', 'stage', 'image_set',
         'session_number', 'session_date', 'plane_name', 'fov_index',
         'targeted_structure', 'imaging_depth', 'frame_rate',
         'scanimage_roi_index', 'genotype', 'name']
planes = (planes[[c for c in order if c in planes.columns]]
          .sort_values(['subject_id', 'session_date', 'fov_index'])
          .reset_index(drop=True))
print(f'{len(planes)} planes across {planes.session_id.nunique()} sessions')
planes.head(10)


## Restrict to what is actually attached

docDB knows about every processed asset; the capsule only mounts some of them. The
problem sets can only open a file that is on `/data`, so filter both tables to the
assets present in the mount &mdash; the same step `bci_metadata.ipynb` does with
`os.listdir`.

If the mount is not attached this cell says so and leaves the tables unfiltered,
rather than silently emitting empty CSVs.


In [ ]:
mount_path = os.path.join(DATA_DIR, CAPSULE_MOUNT)

if os.path.isdir(mount_path):
    attached = set(os.listdir(mount_path))
    print(f'{len(attached)} entries in {CAPSULE_MOUNT}')

    # The mount may be keyed by asset name or by session id -- accept either.
    in_mount = (sessions.name.isin(attached)
                | sessions.session_id.isin(attached)
                | sessions.session_id.map(
                    lambda s: any(a.startswith(s) for a in attached)))
    print(f'{int(in_mount.sum())} of {len(sessions)} sessions present in the mount')

    if in_mount.any():
        sessions['in_capsule'] = in_mount
        planes['in_capsule'] = planes.session_id.isin(
            sessions.loc[in_mount, 'session_id'])
    else:
        print('WARNING: no session names matched the mount contents.')
        print('Sample mount entries:', sorted(attached)[:3])
        sessions['in_capsule'] = False
        planes['in_capsule'] = False
else:
    print(f'{mount_path} not attached to this capsule -- tables not filtered.')
    sessions['in_capsule'] = np.nan
    planes['in_capsule'] = np.nan


## Write the CSVs

`/code/metadata/` rather than `/data/`: the data mounts are read-only inside a
capsule, and these tables are code-adjacent outputs that get versioned with the
notebooks.


In [ ]:
session_csv = os.path.join(OUTPUT_DIR, 'ctl_session_metadata.csv')
plane_csv = os.path.join(OUTPUT_DIR, 'ctl_plane_metadata.csv')

sessions.to_csv(session_csv, index=False)
planes.to_csv(plane_csv, index=False)

print(f'{session_csv}  ({len(sessions)} rows, {sessions.shape[1]} columns)')
print(f'{plane_csv}  ({len(planes)} rows, {planes.shape[1]} columns)')


### Sanity checks before you trust these

docDB drops rows silently &mdash; it returns no error when an asset simply is not
indexed. Read the counts below against what you expect from the processing batch, and
if a mouse is short, re-run its aggregation rather than assuming the data is missing.


In [ ]:
print('sessions per mouse')
print(sessions.subject_id.value_counts().sort_index().to_string())

print('\nplanes per session (should be 8 for most, 2 for late STAGE_1)')
print(planes.groupby('session_id').size().value_counts().to_string())

print('\nsession types')
print(sessions.session_type.value_counts().to_string())

missing_planes = set(sessions.session_id) - set(planes.session_id)
if missing_planes:
    print(f'\n{len(missing_planes)} sessions have no plane rows '
          f'(docDB indexing gap, not necessarily missing data)')
